# 09 — Visualization Overview

A compact tour of every supported `visualize()` family across AbaQuant's
domains: options, portfolios, credit assessments, and market data. Figures
are returned as objects — nothing is displayed automatically (no implicit
`show()`), which keeps this notebook script- and CI-friendly. In Jupyter,
Matplotlib figures render inline automatically when they're the last
expression in a cell.

**Sections:**
1. Option-model figures
2. Portfolio figures
3. Credit figures
4. Market-data figures


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
import pandas as pd

from abaquant.credit.fundamentals import (
    BalanceSheetInputs,
    CashFlowInputs,
    CreditAnalysisInputs,
    IncomeStatementInputs,
    calculate_credit_proxy_metrics,
)
from abaquant.derivatives import OptionStrategy
from abaquant.derivatives.models import BlackScholesMertonModel, CoxRossRubinsteinModel
from abaquant.marketdata import get_ticker, get_tickers
from abaquant.portfolio.optimization import PortfolioAllocator
from abaquant.visualization import VisualizationError

In [ ]:
class DeterministicMarketDataProvider:
    """Minimal offline provider reused across the visualization examples."""

    name = "deterministic-example"

    def fast_info(self, symbol):
        return {"lastPrice": 105.0}

    def info(self, symbol):
        return {"currency": "USD", "marketCap": 600.0, "symbol": symbol}

    def history(self, symbol, **kwargs):
        import numpy as np
        dates = pd.date_range("2025-01-02", periods=24, freq="B")
        return pd.DataFrame({"Close": 100.0 + np.linspace(0, 8, len(dates))}, index=dates)

    def history_many(self, symbols, **kwargs):
        import numpy as np
        dates = pd.date_range("2025-01-02", periods=24, freq="B")
        data = {s: 100.0 + i * 5 + np.linspace(0, 8, len(dates)) for i, s in enumerate(symbols)}
        return pd.DataFrame(data, index=dates)

    def option_expirations(self, symbol):
        return ["2027-01-15"]

    def option_chain(self, symbol, expiry):
        strikes = [80.0, 90.0, 100.0, 110.0, 120.0]
        calls = pd.DataFrame({
            "contractSymbol": [f"{symbol}C{int(k)}" for k in strikes],
            "strike": strikes, "lastPrice": [22.0, 14.5, 8.0, 4.5, 2.4],
            "bid": [21.5, 14.0, 7.6, 4.1, 2.0], "ask": [22.5, 15.0, 8.4, 4.9, 2.8],
            "impliedVolatility": [0.31, 0.27, 0.23, 0.25, 0.29],
            "openInterest": [120, 240, 520, 310, 180], "volume": [12, 28, 65, 34, 16],
        })
        puts = pd.DataFrame({
            "contractSymbol": [f"{symbol}P{int(k)}" for k in strikes],
            "strike": strikes, "lastPrice": [2.2, 4.1, 7.8, 13.9, 21.0],
            "bid": [1.9, 3.8, 7.4, 13.4, 20.4], "ask": [2.5, 4.4, 8.2, 14.4, 21.6],
            "impliedVolatility": [0.35, 0.30, 0.24, 0.26, 0.32],
            "openInterest": [210, 330, 610, 270, 155], "volume": [18, 36, 70, 29, 14],
        })
        return calls, puts

    def income_statement(self, symbol, *, period="annual"):
        return pd.DataFrame({"2025-12-31": {
            "Total Revenue": 450.0, "EBITDA": 90.0, "EBIT": 75.0,
            "Interest Expense": 10.0, "Net Income": 60.0, "Gross Profit": 200.0,
        }})

    def balance_sheet(self, symbol, *, period="annual"):
        return pd.DataFrame({"2025-12-31": {
            "Total Debt": 120.0, "Stockholders Equity": 300.0, "Current Assets": 250.0,
            "Inventory": 40.0, "Current Liabilities": 100.0, "Cash And Cash Equivalents": 50.0,
            "Total Assets": 500.0, "Total Liabilities": 200.0, "Retained Earnings": 110.0,
            "Long Term Debt": 80.0,
        }})

    def cash_flow_statement(self, symbol, *, period="annual"):
        return pd.DataFrame({"2025-12-31": {"Operating Cash Flow": 70.0}})

## 1. Option-model figures

Payoff, price profile, extrinsic value, standardized Greeks, a price
surface, a binomial lattice, and a strategy payoff — all in one pass.


In [ ]:
def build_option_figures():
    model = BlackScholesMertonModel(100.0, 105.0, 1.0, 0.05, 0.20)
    lattice = CoxRossRubinsteinModel(100.0, 105.0, 1.0, 0.05, 0.20, number_of_steps=6)
    return {
        "call_payoff": model.visualize(chart="payoff", option_type="call"),
        "put_profile": model.visualize(chart="price_profile", option_type="put"),
        "call_extrinsic": model.visualize(chart="extrinsic_value", option_type="call"),
        "call_greeks": model.visualize(chart="greeks", option_type="call", greek_scale="standardized"),
        "call_price_surface": model.visualize(
            chart="price_surface", option_type="call", grid_size=31, volatility_grid_size=15
        ),
        "put_lattice": lattice.visualize(chart="tree", option_type="put"),
        "strategy_payoff": OptionStrategy.bull_call_spread(
            lower_strike=100.0, upper_strike=115.0, lower_premium=6.0, upper_premium=2.0
        ).visualize(chart="payoff"),
    }

## 2. Portfolio figures

Weights, cumulative return, and correlation for a maximum-Sharpe allocation.


In [ ]:
def build_portfolio_figures():
    returns = pd.DataFrame(
        {"ALPHA": [0.01, -0.02, 0.03, 0.02], "BETA": [0.005, 0.01, -0.005, 0.003]}
    )
    allocator = PortfolioAllocator(returns, annual_risk_free_rate=0.02)
    weights = allocator.mean_variance.equal_weight()
    return {
        "weights": allocator.visualize(weights=weights, chart="weights"),
        "cumulative_returns": allocator.visualize(weights=weights, chart="cumulative_returns"),
        "correlation": allocator.visualize(chart="correlation"),
    }

## 3. Credit figures

Metric dashboard and synthetic-score visualization.


In [ ]:
def build_credit_figures():
    inputs = CreditAnalysisInputs(
        balance_sheet=BalanceSheetInputs(
            total_debt=100.0, total_equity=200.0, current_assets=120.0, current_liabilities=60.0
        ),
        income_statement=IncomeStatementInputs(ebit=50.0, ebitda=60.0, interest_expense=5.0),
        cash_flow_statement=CashFlowInputs(operating_cash_flow=40.0),
    )
    assessment = calculate_credit_proxy_metrics(inputs)
    return {
        "credit_metrics": assessment.visualize(chart="metrics"),
        "credit_score": assessment.visualize(chart="score"),
    }

## 4. Market-data figures

Ticker price history, one financial statement, and universe price history.


In [ ]:
def build_marketdata_figures():
    provider = DeterministicMarketDataProvider()
    ticker = get_ticker("DEMO", provider=provider, financial_cache="memory")
    universe = get_tickers(["ALPHA", "BETA", "GAMMA"], provider=provider)
    return {
        "ticker_history": ticker.visualize(period="1mo"),
        "financial_statement": ticker.financials.visualize(statement="balance_sheet"),
        "universe_history": universe.visualize(period="1mo"),
    }

## Build everything and summarize


In [ ]:
try:
    all_figures = {}
    for group in (
        build_option_figures(),
        build_portfolio_figures(),
        build_credit_figures(),
        build_marketdata_figures(),
    ):
        all_figures.update(group)
    print(f"Created {len(all_figures)} figures total.")
    for name, fig in all_figures.items():
        print(f"  {name:24s}: {type(fig).__name__}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

Every domain facade in AbaQuant follows the same `.visualize(chart=...)`
convention. See notebook **11 — Visualize Method Gallery** for an even more
exhaustive walkthrough, and **10 — Visualization Theme** for global styling
and export control.
